# SparkLab 2026–27 — From RDDs to Optimized DataFrames
**Advanced Database Topics · 3-hour guided lab + optional self-study**

Execute paragraphs **top to bottom** using Spark 4.2.x. The employee CSV is the only external input. Extra demonstration tables are created in memory. Sections marked **[OPTIONAL]** are extensions, not essential for the live session.

**Learning path:** transformations/actions → partitioning/shuffles → relational analytics → joins/windows → Parquet/explain/UI → VARIANT/vector/nearest-neighbors. Examples illustrate the semester project methods without solving its queries.


## 0. Session setup (5 min)
`%livy.pyspark` creates/uses a Livy Spark session; not necessary in terminal mode. Set `EMPLOYEES_PATH` to a path visible to executors (prefer HDFS for multi-node clusters). Run the following paragraph once.

In a standalone `spark-submit` application, the driver/session configuration belongs in the submission command, not in individual DataFrame operations. Compare worker capacity vs application executor settings in the assignment.


In [2]:
%livy.pyspark
from pyspark.sql import functions as F, Window
from pyspark.sql import types as T
import time

from pyspark.sql import SparkSession

sc = SparkSession \
    .builder \
    .appName("Session Setup") \
    .getOrCreate() \
    .sparkContext

print('Spark version:', spark.version)
print('Master:', spark.sparkContext.master)
print('Default parallelism:', spark.sparkContext.defaultParallelism)
print('Shuffle partitions:', spark.conf.get('spark.sql.shuffle.partitions'))
print('AQE enabled:', spark.conf.get('spark.sql.adaptive.enabled'))
# EDIT THIS path for your lab environment; the file must be accessible to the cluster.
EMPLOYEES_PATH = 'hdfs://nchalv-1:54310/user/ubuntu/ta/advanced-db/spark-examples/employees_7col.csv'


## 1. RDDs: distributed data and lazy evaluation (20 min)
Predict which lines trigger a job. Narrow `map`/`filter` can be pipelined; `reduceByKey` causes a shuffle. Use the **Jobs / Stages** tabs after running.


In [4]:
%livy.pyspark
sc = spark.sparkContext
lines = sc.parallelize(['Spark is fast', 'Spark SQL is useful', 'SQL is expressive'], 3)
words = lines.flatMap(lambda line: line.lower().split())
counts = words.map(lambda w: (w, 1)).reduceByKey(lambda a, b: a + b)
print('Partitions:', counts.getNumPartitions())
print('Lineage:', counts.toDebugString().decode('utf-8', errors='replace') if isinstance(counts.toDebugString(), bytes) else counts.toDebugString())
print('Result:', sorted(counts.collect()))  # ACTION


## Example 1 — RDDs, Lazy Evaluation & Shuffle

This simple **Word Count** example demonstrates five fundamental concepts of Apache Spark:

1. **Data partitioning and parallelism**
2. **Lazy evaluation**
3. **Narrow vs. wide transformations**
4. **Shuffling and stage boundaries**
5. **RDD lineage and fault tolerance**

### 1. Data Partitioning

```python
lines = sc.parallelize([
    "Spark is fast",
    "Spark SQL is useful",
    "SQL is expressive"
], 3)
```

![Partitions](figures/partitions.png)

The input is divided into **3 partitions**, which can be processed independently by Spark tasks.

- Each partition contains one input line.
- Each partition is processed by a task.
- Up to 3 tasks can execute simultaneously, provided sufficient executor cores are available.
- **Partitions are not executors:** Multiple partitions can be processed sequentially or concurrently by the same executor.

The number of partitions influences the available parallelism.

### 2. Narrow Transformations: flatMap() and map()

```python
words = lines.flatMap(lambda line: line.lower().split())
pairs = words.map(lambda w: (w, 1))
```

Both are **narrow transformations**:

- `flatMap()` splits each line into words.
- `map()` converts each word into a `(word, 1)` pair.
- Neither operation requires exchanging data between partitions.

Spark can pipeline these transformations into the same execution stage.

### 3. Wide Transformation: reduceByKey()

```python
counts = pairs.reduceByKey(lambda a, b: a + b)
```

`reduceByKey()` is a **wide transformation** because values belonging to the same key may reside in different partitions.

For example, the word `spark` appears in two input partitions. Spark must combine the partial counts to produce the global result.

The operation typically involves:

1. **Local aggregation:** Combine values for the same key within each input partition.
2. **Shuffle:** Redistribute intermediate results so that values for the same key reach the same destination partition.
3. **Final aggregation:** Combine the received values to produce the global count.

A shuffle requires additional communication and usually creates a **stage boundary**.

![Shuffle](figures/shuffle.png)

**Key observation:** Operations such as `groupBy()`, `reduceByKey()`, joins and window functions can require expensive shuffles.

### 4. RDD Lineage

Calling `counts.toDebugString()` produces a dependency graph similar to:

```text
(3) PythonRDD
 |  MapPartitionsRDD
 |  ShuffledRDD
 +-(3) PairwiseRDD
    |  PythonRDD
    |  ParallelCollectionRDD
```

Interpretation:

- `ParallelCollectionRDD`: Original distributed collection.
- `PythonRDD`: Python transformations applied to the data.
- `PairwiseRDD`: Intermediate key-value representation.
- `ShuffledRDD`: Data redistribution associated with `reduceByKey()`.
- `MapPartitionsRDD`: Internal transformation used in producing the final RDD.

The vertical `|` connectors represent **narrow dependencies**, while `+-` identifies a **shuffle dependency**.

**Fault tolerance:** Spark records the transformations used to construct an RDD. If a partition is lost, Spark can use this lineage to recompute the missing data.

### 5. Lazy Evaluation: Transformations vs. Actions

Spark does not immediately execute the distributed computation when transformations are defined:

```python
words = lines.flatMap(...)
pairs = words.map(...)
counts = pairs.reduceByKey(...)
```

These operations construct the RDD lineage.

Likewise:

```python
counts.getNumPartitions()
counts.toDebugString()
```

inspect metadata without triggering execution of the word count.

Execution begins when an **action** is invoked:

```python
counts.collect()
```

At this point, Spark schedules tasks, executes the transformations, performs the necessary shuffle and sends the final results to the driver.

**Important:** `collect()` transfers all result records to the driver. It should only be used when the result is sufficiently small.

### 6. Experiment: Changing Shuffle Partitions

Try modifying the number of output partitions:

```python
counts2 = (
    words.map(lambda w: (w, 1))
         .reduceByKey(lambda a, b: a + b, numPartitions=2)
)

print("Input partitions:", lines.getNumPartitions())
print("Output partitions:", counts2.getNumPartitions())
print("Result:", sorted(counts2.collect()))
```

Expected:

```text
Input partitions: 3
Output partitions: 2
```

The word frequencies remain unchanged.

However, the execution topology changes:

- The shuffle map stage processes **3 input partitions**.
- The subsequent reduce stage processes **2 output partitions**.

Inspect the **Spark UI → Jobs → Stages** to observe the number of tasks and shuffle metrics.

### Takeaways

- Spark processes data in **partitions**.
- **Narrow transformations** can execute without redistributing data.
- **Wide transformations** introduce shuffle dependencies and generally separate execution stages.
- **Lazy evaluation** postpones computation until an action is invoked.
- **Lineage** provides the information needed to recompute lost partitions.
- The number of partitions affects execution parallelism, but does not necessarily change query results.

**Discussion question:** If a Spark application has 4 available executor cores but an RDD contains only 2 partitions, how many tasks can execute concurrently in a stage processing that RDD?


**Ask:** Why is `reduceByKey` generally preferable to `groupByKey` for word counts? What changes if the input has 1 vs 10 partitions? Why must `collect()` be avoided on unbounded output? RDDs are instructive; the semester project uses **DataFrame and SQL APIs**.


## 2. CSV ingestion and explicit schemas (15 min)
Spark may infer incorrect types or incur an extra scan. A declared schema is reproducible. The provided employee CSV uses uppercase headers, integer salaries, and timestamp strings with timezone offsets.


In [8]:
%livy.pyspark
employee_schema = T.StructType([
    T.StructField('ID', T.IntegerType()),
    T.StructField('FIRST_NAME', T.StringType()),
    T.StructField('LAST_NAME', T. StringType()),
    T.StructField('EMAIL', T.StringType()),
    T.StructField('HIRE_DATE', T.StringType()),
    T.StructField('SALARY', T.IntegerType()),
    T.StructField('DEPARTMENT', T.StringType()),
])
employees = (spark.read.option('header', True).schema(employee_schema)
             .csv(EMPLOYEES_PATH))
employees.printSchema()
employees.show(6, truncate=False)
print('Total employees:', employees.count())
employees.createOrReplaceTempView('employees')


## Example 2 — DataFrames, Schemas & Lazy Evaluation

This example introduces the **Spark DataFrame API**, the primary interface used throughout the semester assignment.

A DataFrame is a distributed collection of structured records organized into named columns with known data types.

### 1. Explicit Schemas

CSV files contain textual representations of values. By defining a schema explicitly, we specify how Spark should interpret each column:

- `IntegerType()` for numeric identifiers and salaries.
- `StringType()` for textual attributes.
- `StructType()` and `StructField()` to describe the complete record structure.

**Why not use `inferSchema=True`?**

Schema inference requires examining the input data to determine appropriate types. Explicit schemas avoid this overhead and make data ingestion more predictable.

**Observation:** The schema shows `nullable = true` because fields are nullable by default. This does not necessarily mean that null values exist.

### 2. Lazy Evaluation and Actions

Creating a DataFrame defines a logical computation. Distributed execution generally begins when an **action** is invoked.

| Operation | Spark job? | Purpose |
|-----------|------------|---------|
| `printSchema()` | No | Inspect column names and types |
| `show(6)` | Yes | Retrieve and display rows |
| `count()` | Yes | Count all records |
| `createOrReplaceTempView()` | No | Register a temporary SQL view |

**Important:** `show()` and `count()` trigger separate computations. Spark does not automatically cache intermediate results.

### 3. DataFrames vs. RDDs

| RDD API | DataFrame API |
|---------|---------------|
| General distributed objects | Structured rows and columns |
| Explicit `map`, `filter`, `reduceByKey` | Relational `select`, `filter`, `groupBy`, `join` |
| Limited automatic optimization | Catalyst query optimization |
| Python functions often execute in Python workers | Built-in relational operations generally execute in the JVM |
| Programmer controls transformations | Spark can optimize the execution plan |

**Key difference:** DataFrames expose the structure and meaning of operations, allowing Spark to optimize their execution.

### 4. SQL Integration

```python
employees.createOrReplaceTempView("employees")
```

Registers a session-scoped temporary view that can be queried using Spark SQL:

```sql
SELECT DEPARTMENT, AVG(SALARY) AS avg_salary
FROM employees
GROUP BY DEPARTMENT
ORDER BY avg_salary DESC
```

DataFrame operations and SQL queries are processed by the same underlying Spark SQL engine.

A temporary view does **not** create a physical table or automatically store data in memory.

### 5. Execution and Parallelism

Calling:

```python
employees.count()
```

triggers a Spark job that computes the total number of records.

Spark can calculate partial counts across input partitions and combine them into a final result.

Even though this dataset contains only **19 employees**, it is processed using Spark's distributed execution model.

The actual number of tasks depends on input partitioning and the selected execution plan.

### Takeaways

- DataFrames provide **structured, schema-aware distributed processing**.
- Explicit schemas improve predictability and avoid schema-inference overhead.
- DataFrame transformations are **lazily evaluated**.
- Actions such as `show()` and `count()` trigger computation.
- Spark SQL and the DataFrame API share the **Catalyst optimizer**.
- Temporary SQL views provide an alternative query interface without materializing data.

### Questions for Discussion

1. Why does `printSchema()` not require reading every record?
2. Why can `show(6)` and `count()` trigger different Spark jobs?
3. Would a DataFrame containing 19 records necessarily execute 19 tasks?
4. What happens if a CSV field contains an invalid value for its declared type?



In [10]:
%livy.pyspark
setattr(sc._jvm, "PythonSQLUtils", sc._jvm.org.apache.spark.sql.api.python.PythonSQLUtils) #normally you won't need this line

print('Partitions:', employees.rdd.getNumPartitions())
employees.explain(mode='formatted')

## 3. DataFrame operations: filter, select, derive, aggregate (20 min)
These operations correspond to the filter/projection/aggregation patterns in the project. Observe that `show()` is an action and that a `DataFrame` transformation returns a **new** DataFrame.


In [12]:
%livy.pyspark
high_paid = (employees
    .where(F.col('SALARY') >= 5000)
    .select('ID', 'FIRST_NAME', 'DEPARTMENT', 'SALARY')
    .orderBy(F.desc('SALARY'), 'ID'))
high_paid.show(truncate=False)
by_department = (employees.groupBy('DEPARTMENT')
    .agg(F.count('*').alias('n'),
         F.round(F.avg('SALARY'), 2).alias('avg_salary'),
         F.max('SALARY').alias('max_salary'))
    .orderBy(F.desc('avg_salary')))
by_department.show()


## Example 3 — DataFrame Transformations: Filter, Select, Derive & Aggregate

The PySpark DataFrame API provides high-level relational operations for processing structured data.

Unlike low-level RDD transformations, DataFrame operations expose their semantics to the **Catalyst optimizer**, allowing Spark to optimize the entire query pipeline.

### 1. Filtering — `filter()` / `where()`

```python
employees.where(F.col("SALARY") >= 5000)
```

Filtering retains only records satisfying a Boolean condition.

- `filter()` and `where()` are equivalent.
- Conditions can combine comparisons using `&` (AND), `|` (OR) and `~` (NOT).
- Filtering does not modify the original DataFrame.
- Filtering is generally a **narrow transformation**.

Example:

```python
employees.filter(
    (F.col("SALARY") >= 5000) &
    (F.col("DEPARTMENT") == "IT")
).show()
```

### 2. Projection — `select()`

```python
employees.select("ID", "FIRST_NAME", "SALARY")
```

Projection determines which columns appear in the result.

Spark's optimizer can remove unnecessary columns from the scan through **column pruning**.

Filtering and projection can be composed:

```python
employees.where(F.col("SALARY") >= 5000) \
         .select("FIRST_NAME", "SALARY")
```

Both operations are generally narrow and can be pipelined within the same execution stage.

### 3. Deriving New Columns — `withColumn()`

New attributes can be computed using expressions:

```python
derived = (
    employees
    .withColumn("ANNUAL_SALARY", F.col("SALARY") * 12)
    .withColumn(
        "SALARY_LEVEL",
        F.when(F.col("SALARY") >= 7000, "High")
         .when(F.col("SALARY") >= 4000, "Medium")
         .otherwise("Low")
    )
)

derived.select(
    "FIRST_NAME", "SALARY", "ANNUAL_SALARY", "SALARY_LEVEL"
).show()
```

- `withColumn()` adds or replaces a column.
- `when()` / `otherwise()` implements conditional expressions, similar to SQL `CASE WHEN`.
- The original DataFrame remains unchanged.

**Project connection:** Query 1 requires assigning reviews to length categories using conditional expressions.

### 4. Grouping and Aggregation — `groupBy()` / `agg()`

```python
by_department = (
    employees.groupBy("DEPARTMENT")
    .agg(
        F.count("*").alias("n"),
        F.round(F.avg("SALARY"), 2).alias("avg_salary"),
        F.max("SALARY").alias("max_salary")
    )
)
```

Grouping changes the granularity of the dataset:

**Before:** One row per employee.

**After:** One row per department.

Aggregation functions include:

- `count()` — number of records or non-null values
- `sum()` — total
- `avg()` / `mean()` — arithmetic mean
- `min()` / `max()` — minimum and maximum
- `countDistinct()` — number of distinct values
- `stddev()` — standard deviation

**Important:** `count("*")` counts all rows, while `count("SALARY")` ignores rows whose salary is NULL.

A distributed `groupBy()` typically involves:

1. Local partial aggregations within partitions.
2. A **shuffle** to bring together records belonging to the same group.
3. Final aggregation to obtain the global result.

Spark can optimize many built-in aggregations by combining values locally before shuffling.

### 5. Sorting — `orderBy()` / `sort()`

```python
employees.orderBy(F.desc("SALARY"), F.asc("ID"))
```

Sorting determines the ordering of the final result.

- `asc()` sorts in ascending order.
- `desc()` sorts in descending order.
- Multiple columns can define tie-breaking rules.
- Global sorting across multiple partitions generally requires a shuffle and sorting work.

**Important:** Without an explicit `orderBy()`, the row order of a distributed DataFrame is not guaranteed.

### 6. Transformations vs. Actions

All the following construct new DataFrames without immediately computing their results:

```python
df2 = employees.filter("SALARY > 5000")
df3 = df2.select("FIRST_NAME", "SALARY")
df4 = df3.orderBy("SALARY")
```

Execution occurs when an action is invoked:

```python
df4.show()
```

Spark can optimize the complete sequence rather than executing each transformation independently.

### 7. Inspecting the Physical Plan

Compare the execution plans:

```python
high_paid.explain(mode="formatted")
by_department.explain(mode="formatted")
```

Look for the following operators:

| Physical operator | Meaning |
|-------------------|---------|
| `Scan` | Read the input data |
| `Filter` | Remove rows that do not satisfy a condition |
| `Project` | Select or compute columns |
| `HashAggregate` | Perform grouped aggregation |
| `Sort` | Order records |
| `Exchange` | Redistribute data between partitions |

**Note:** The precise operators depend on the optimizer, input partitioning and execution configuration.

### Takeaways

- Filtering, projection and derivation are generally **narrow transformations**.
- Grouping and global sorting typically require **distributed coordination and shuffling**.
- DataFrame transformations are **lazy** and immutable.
- Catalyst can optimize the complete transformation pipeline.
- Explicit sorting is necessary when the order of the results matters.

### Questions for Discussion

1. Why can Spark apply filtering and projection without a shuffle?
2. Why does grouping by department potentially require exchanging records between workers?
3. What is the difference between `count("*")` and `count("SALARY")`?
4. Why is it useful to compute partial aggregations before the shuffle?
5. Why should we avoid assuming that a DataFrame has an inherent row order?


In [14]:
%livy.pyspark
high_paid.explain(mode="formatted")
by_department.explain(mode="formatted")

### Same query in Spark SQL
For a fair DataFrame-vs-SQL comparison, compare results **and optimized physical plans**. Equivalent statements often optimize to the same plan.


In [16]:
%livy.pyspark
sql_department = spark.sql('''SELECT DEPARTMENT, COUNT(*) AS n,
       ROUND(AVG(SALARY), 2) AS avg_salary, MAX(SALARY) AS max_salary
FROM employees GROUP BY DEPARTMENT ORDER BY avg_salary DESC''')
sql_department.show()
print('DataFrame plan:')
by_department.explain(mode='formatted')
print('SQL plan:')
sql_department.explain(mode='formatted')


## Example 3 — Spark SQL vs. DataFrame API: Physical Plans

The same relational query can be expressed using either the **DataFrame API** or **Spark SQL**.

Both approaches use the Spark SQL execution engine, including the Catalyst optimizer.

### 1. Equivalent Queries

**DataFrame API**

```python
by_department = (
    employees.groupBy("DEPARTMENT")
    .agg(
        F.count("*").alias("n"),
        F.round(F.avg("SALARY"), 2).alias("avg_salary"),
        F.max("SALARY").alias("max_salary")
    )
    .orderBy(F.desc("avg_salary"))
)
```

**Spark SQL**

```sql
SELECT DEPARTMENT,
       COUNT(*) AS n,
       ROUND(AVG(SALARY), 2) AS avg_salary,
       MAX(SALARY) AS max_salary
FROM employees
GROUP BY DEPARTMENT
ORDER BY avg_salary DESC
```

Both produce identical results and, in this example, equivalent physical execution plans.

**Key observation:** The physical plan is determined primarily by the required computation, not by whether the query was written using SQL or the DataFrame API.

### 2. Understanding the Physical Plan

![Physical plan](figures/physycal_plan.png)


Read the following plan **from bottom to top**:

```text
AdaptiveSparkPlan
+- Sort
   +- Exchange (range partitioning)
      +- HashAggregate (final)
         +- Exchange (hash partitioning)
            +- HashAggregate (partial)
               +- Scan csv
```

| Operator | Role |
|----------|------|
| `Scan csv` | Read the relevant input columns |
| `HashAggregate (partial)` | Compute local intermediate aggregates |
| `Exchange (hash)` | Redistribute partial aggregates by DEPARTMENT |
| `HashAggregate (final)` | Combine intermediate aggregates |
| `Exchange (range)` | Redistribute results for global ordering |
| `Sort` | Sort the results by average salary |
| `AdaptiveSparkPlan` | Enable adaptive optimization during execution |

### 3. Partial vs. Final Aggregation

Spark commonly performs aggregations in two phases:

**Partial aggregation:** Each input partition computes intermediate values.

For `AVG(SALARY)`, these are the sum and count of salaries.

**Final aggregation:** After a shuffle, intermediate results are combined.

The global average is:


`AVG(x) = SUM(partial_sums) / SUM(partial_counts)`


This reduces communication compared with shuffling every original record.

### 4. Understanding the Two Shuffles

The physical plan includes two `Exchange` operators:

**First Exchange — Hash Partitioning**

```text
hashpartitioning(DEPARTMENT, 200)
```

Ensures that records belonging to the same department reach the same destination partition for final aggregation.

**Second Exchange — Range Partitioning**

```text
rangepartitioning(avg_salary DESC, 200)
```

Redistributes aggregated results into ordered ranges to support the global `ORDER BY`.

**Important:** A shuffle may require substantial network I/O, serialization and disk spilling when working with large datasets.

### 5. Shuffle Partitions

The value `200` is normally determined by:

```python
spark.conf.get("spark.sql.shuffle.partitions")
```

It represents the configured number of shuffle partitions, not the number of executors or CPU cores.

With **Adaptive Query Execution (AQE)**, Spark can coalesce small shuffle partitions during execution.

### 6. Adaptive Query Execution

```text
AdaptiveSparkPlan
Arguments: isFinalPlan=false
```

Indicates that AQE is enabled and the displayed plan is not yet finalized.

AQE can use runtime statistics to optimize execution, including:

- Coalescing small shuffle partitions.
- Changing certain join strategies.
- Handling skewed partitions in supported scenarios.

**Distinction:**
- Catalyst optimizes the query before execution.
- AQE can revise the physical plan during execution.

### 7. Column Pruning

Notice the scan:

```text
ReadSchema: struct<SALARY:int,DEPARTMENT:string>
```

Although the input CSV has seven columns, the query only requires two.

Catalyst therefore eliminates unnecessary columns from the scan.

This is called **column pruning**.

### Experiment: Remove ORDER BY

```python
without_sort = (
    employees.groupBy("DEPARTMENT")
    .agg(F.avg("SALARY").alias("avg_salary"))
)

without_sort.explain("formatted")
```

Compare the execution plan with the original query.

**Expected observation:** The global `Sort` and range-partitioning `Exchange` disappear.

### Takeaways

- DataFrame and SQL queries can produce **equivalent physical plans**.
- Spark often performs aggregation in **partial and final phases**.
- `Exchange` indicates data redistribution and usually a shuffle boundary.
- Grouping and global sorting impose different partitioning requirements.
- `spark.sql.shuffle.partitions` controls the initial shuffle partition count.
- AQE can adapt physical execution to runtime data characteristics.
- Column pruning minimizes unnecessary input processing.

### Discussion Questions

1. Why does Spark use two `HashAggregate` operators?
2. Why are partial sums and counts sufficient to compute a global average?
3. What is the difference between hash and range partitioning?
4. Why does removing `ORDER BY` eliminate a shuffle?
5. Does seeing `200` shuffle partitions imply that 200 tasks will necessarily execute?
6. If SQL and DataFrame APIs produce the same physical plan, should their execution times necessarily be identical?


## 4. Relational joins, NULL handling, and outer/anti joins (20 min)
An additional dimension table is constructed in the notebook. One department intentionally has no matching row. Discuss inner vs left join; `left_anti` identifies employees with **no matching department** (analogous to identifying unmatched records in project queries).


In [19]:
%livy.pyspark
departments = spark.createDataFrame([
    ('Engineering', 'Technical', 1), ('Sales', 'Commercial', 2),
    ('Marketing', 'Commercial', 3), ('IT', 'Technical', 4),
    ('QA', 'Technical', 5)
], ['DEPARTMENT', 'DIVISION', 'COST_CENTER'])
departments.createOrReplaceTempView('departments')
joined = employees.join(departments, 'DEPARTMENT', 'left')
joined.select('FIRST_NAME', 'DEPARTMENT', 'DIVISION').show(30, truncate=False)
print('Unmatched employees:')
employees.join(departments, 'DEPARTMENT', 'left_anti').show(truncate=False)
print('Physical plan:')
joined.explain(mode='formatted')


## Example 4 — Distributed Joins, Anti-Joins & Broadcast Optimization

Joins combine records from multiple datasets based on matching keys. They are fundamental relational operations but can be among the most expensive operations in distributed data processing.

### 1. Left Outer Join

```python
joined = employees.join(
    departments,
    "DEPARTMENT",
    "left"
)
```

A **left outer join** preserves all rows from the left DataFrame.

- Matching departments contribute their `DIVISION` and `COST_CENTER`.
- Employees without a matching department remain in the result.
- Missing right-hand attributes are represented by `NULL`.

In our example, the three employees from `Security` have no matching department, so their `DIVISION` is `NULL`.

### 2. Left Anti Join

```python
unmatched = employees.join(
    departments,
    "DEPARTMENT",
    "left_anti"
)
```

A **left anti join** returns rows from the left dataset for which no match exists in the right dataset.

It is equivalent in intent to a SQL `NOT EXISTS` query:

```sql
SELECT e.*
FROM employees e
WHERE NOT EXISTS (
    SELECT 1
    FROM departments d
    WHERE e.DEPARTMENT = d.DEPARTMENT
)
```

In this example, the result consists of the three employees belonging to `Security`.

**Project connection:** Anti-joins are useful for identifying users with no reviews and records that have no match in another dataset.

### 3. Sort-Merge Join (SMJ)

The default physical plan selected by Spark contains:

```text
AdaptiveSparkPlan
+- Project
   +- SortMergeJoin LeftOuter
      :- Sort
      :  +- Exchange
      :     +- Scan csv
      +- Sort
         +- Exchange
            +- Filter
               +- Scan ExistingRDD
```

A **Sort-Merge Join** works by:

1. Redistributing both datasets by their join key.
2. Sorting records by that key within each partition.
3. Merging the sorted inputs to identify matching records.

![Sort-Merge Join](figures/sortmerge.png)


The plan contains:

- **Two `Exchange` operators:** Shuffle both datasets by `DEPARTMENT`.
- **Two `Sort` operators:** Sort the redistributed partitions by join key.
- **One `SortMergeJoin`:** Match records from the sorted inputs.

Both sides use compatible hash partitioning:

```text
hashpartitioning(DEPARTMENT, 200)
```

The `200` corresponds to the configured shuffle partition count, which AQE may reduce at runtime.

### 4. Broadcast Hash Join (BHJ)

When one dataset is sufficiently small, Spark can send a copy of it to the executors processing the larger dataset.

This is called a **Broadcast Hash Join**.

```python
broadcast_joined = employees.join(
    F.broadcast(departments),
    "DEPARTMENT",
    "left"
)

broadcast_joined.explain("formatted")
```

The resulting physical plan should contain:

```text
BroadcastHashJoin LeftOuter BuildRight
```

and a `BroadcastExchange` operator for the departments table.

**Advantages:**
- Avoids repartitioning both inputs by the join key.
- Avoids sorting both datasets.
- Often improves performance for large fact tables joined with small lookup tables.

**Limitations:**
- The broadcast dataset must fit comfortably in executor memory.
- Broadcasting excessively large datasets can cause memory pressure or failures.
- A broadcast hint does not guarantee that every join configuration can use a broadcast hash join.

### 5. Why Did Catalyst Initially Choose Sort-Merge Join?

Although our departments table contains only five records, Spark initially selected Sort-Merge Join.

The DataFrame was constructed from a Python collection and may not have reliable size statistics available during query planning.

Spark therefore does not necessarily recognize it as an ideal broadcast candidate.

An explicit `F.broadcast()` hint communicates this preference to the optimizer.

### 6. Join Semantics vs. Physical Strategy

It is important to distinguish:

**Logical join type:** What records should be returned?

- `inner`
- `left`
- `right`
- `full`
- `left_semi`
- `left_anti`

**Physical join strategy:** How does Spark compute the join?

- Broadcast Hash Join (BHJ)
- Sort-Merge Join (SMJ)
- Shuffle Hash Join (SHJ)
- Broadcast Nested Loop Join (BNLJ)

Different physical strategies can implement the same logical join operation.

### 7. Experiment: Compare Join Plans

```python
join_default = employees.join(
    departments, "DEPARTMENT", "left"
)

join_broadcast = employees.join(
    F.broadcast(departments), "DEPARTMENT", "left"
)

join_anti = employees.join(
    departments, "DEPARTMENT", "left_anti"
)

for name, df in [
    ("Default", join_default),
    ("Broadcast", join_broadcast),
    ("Anti", join_anti)
]:
    print(name)
    df.explain("formatted")
```

Compare the physical plans and identify:

1. Which plan contains `SortMergeJoin`?
2. Which plan contains `BroadcastHashJoin`?
3. Which plan contains `BroadcastExchange`?
4. Which plan requires shuffling both datasets?
5. Why does the anti-join return fewer records?

### Takeaways

- Joins combine distributed datasets using matching keys.
- A **left outer join** preserves unmatched left-side records.
- A **left anti join** returns records without a match.
- **Sort-Merge Join** typically requires shuffling and sorting both sides.
- **Broadcast Hash Join** avoids shuffling the large side when the smaller side can be broadcast.
- Catalyst selects physical join strategies based on available statistics, join semantics and optimization rules.
- AQE may revise join strategies using runtime statistics.

### Discussion Questions

1. Why are two `Exchange` operators needed for the original Sort-Merge Join?
2. Why is broadcasting a five-row lookup table preferable to shuffling millions of records?
3. What could happen if we broadcast a dataset containing several gigabytes?
4. Does a left anti join return columns from both datasets?
5. Can two queries with identical join semantics use different physical execution strategies?


### [OPTIONAL] Explicit join hints
Use a hint **only** after checking table sizes and `explain()`. Broadcast is attractive for a genuinely small dimension table; it is not a universal optimization.


In [22]:
%livy.pyspark
print('Default join')
employees.join(departments, 'DEPARTMENT').explain(mode='formatted')
print('Broadcast-hinted join')
employees.join(F.broadcast(departments), 'DEPARTMENT').explain(mode='formatted')


## Example 4 — Join Hints & Optimizer Decisions

Spark's Catalyst optimizer selects a physical join strategy based on:

- Join type and join condition.
- Estimated input sizes and available statistics.
- Physical properties of the input datasets.
- Spark configuration and optimization rules.
- Explicit optimization hints.

A **join hint** allows the programmer to express a preference for a particular physical execution strategy.

### 1. Default Join vs. Broadcast Hint

**Default inner join:**

```python
employees.join(
    departments, "DEPARTMENT"
).explain("formatted")
```

**Broadcast-hinted inner join:**

```python
employees.join(
    F.broadcast(departments), "DEPARTMENT"
).explain("formatted")
```

Both queries produce equivalent results, but the physical plans differ.

| Property | Default | Broadcast-hinted |
|----------|---------|------------------|
| Physical join | BroadcastHashJoin | BroadcastHashJoin |
| Build side | Left | Right |
| Broadcast dataset | employees | departments |
| Shuffle required | No | No |

### 2. Understanding BuildLeft and BuildRight

The default plan contains:

```text
BroadcastHashJoin Inner BuildLeft
```

Spark chooses to broadcast the left input (`employees`).

The hinted plan contains:

```text
BroadcastHashJoin Inner BuildRight
```

Spark instead broadcasts the right input (`departments`).

**Why did Spark broadcast 19 employees instead of 5 departments?**

Catalyst selects strategies using **estimated data sizes**, not necessarily actual row counts.

The employees DataFrame is backed by an HDFS file with available size metadata, whereas the departments DataFrame was constructed programmatically and may lack accurate size statistics.

Consequently, the optimizer can select a strategy that seems counterintuitive when considering only the actual record counts.

### 3. How Broadcast Hash Join Works

A Broadcast Hash Join:

1. Constructs a hash table from the smaller or selected **build-side** dataset.
2. Broadcasts the hash table to executors.
3. Processes partitions of the other dataset locally, probing the hash table for matches.


![Broadcast Hash Join](figures/broadcast_hash.png)

This avoids shuffling and sorting both input datasets by the join key.

Broadcast joins are particularly effective for **large fact tables joined with small lookup tables**.

### 4. Inspecting Estimated Statistics

```python
employees.explain(mode="cost")
departments.explain(mode="cost")
```

Look for:

```text
Statistics(sizeInBytes=...)
```

These estimates help explain optimizer decisions.

**Important:** Estimated sizes may differ substantially from actual dataset sizes.

### 5. Other Join Hints

Spark provides several join strategy hints.

| Hint | Preferred strategy |
|------|--------------------|
| `broadcast` | Broadcast Hash Join, when applicable |
| `merge` | Sort-Merge Join |
| `shuffle_hash` | Shuffle Hash Join |
| `shuffle_replicate_nl` | Shuffle-and-Replicate Nested Loop Join |

Examples:

```python
employees.join(
    departments.hint("merge"), "DEPARTMENT"
).explain("formatted")

employees.join(
    departments.hint("shuffle_hash"), "DEPARTMENT"
).explain("formatted")
```

**Caution:** Hints express preferences. Spark may ignore a hint if the requested strategy cannot support the join.

### 6. Join Type Matters

The previous example used a left outer join:

```python
employees.join(departments, "DEPARTMENT", "left")
```

The current example uses an inner join (the default):

```python
employees.join(departments, "DEPARTMENT")
```

Inner equi-joins can build a broadcast hash table from either input.

Left outer equi-joins can use a Broadcast Hash Join with the **right side as the build side**, preserving unmatched rows from the left.

Therefore, join semantics also constrain the available physical strategies.

### 7. When Should We Use Hints?

**Prefer the optimizer's default decisions initially.**

Consider hints when:

- The optimizer has inaccurate or missing statistics.
- A small lookup dataset should clearly be broadcast.
- Experiments demonstrate that a different strategy performs better.
- You need to compare physical strategies for performance analysis.

Avoid blindly forcing join strategies without examining memory requirements and execution costs.

### Takeaways

- Join hints influence **physical execution**, not query semantics.
- `BuildLeft` and `BuildRight` identify which input builds the hash table.
- Broadcast joins avoid shuffling the probe-side dataset.
- Optimizer decisions depend on **estimated sizes**, not simply record counts.
- Logical join types constrain valid physical join strategies.
- Hints should be justified using physical plans and experimental evidence.

### Discussion Questions

1. Why did Spark initially broadcast the employees dataset?
2. Why does `F.broadcast(departments)` change the build side?
3. What is the difference between a broadcast dataset and a shuffled dataset?
4. Why is broadcasting an excessively large dataset dangerous?
5. Why can't a left outer Broadcast Hash Join build the left side?
6. Should optimization hints always improve query performance?


## 5. Window functions and top-k per group (15 min)
A `LIMIT 3` gives the global top three; `row_number()` with `partitionBy()` gives the top three **within each department**. Add a deterministic tie-breaker (ID). This pattern is essential to the project's state/category ranking query.


In [25]:
%livy.pyspark
w = Window.partitionBy('DEPARTMENT').orderBy(F.desc('SALARY'), F.asc('ID'))
top2 = (employees.withColumn('rn', F.row_number().over(w))
        .where(F.col('rn') <= 2)
        .select('DEPARTMENT', 'FIRST_NAME', 'SALARY', 'rn')
        .orderBy('DEPARTMENT', 'rn'))
top2.show(50, truncate=False)
top2.explain(mode='formatted')


## Example 5 — Window Functions & Top-k per Group

Window functions allow us to compute rankings, aggregates and other analytical values **within groups of records without collapsing those records into a single row per group**.

This makes them particularly useful for implementing **top-k queries**, such as finding the highest-paid employees in each department.

### 1. Defining a Window

```python
w = Window.partitionBy("DEPARTMENT") \
          .orderBy(F.desc("SALARY"), F.asc("ID"))
```

The window specification defines:

- **PARTITION BY:** Divide employees into logical groups by department.
- **ORDER BY:** Sort employees within each department by decreasing salary.
- **Tie-breaking:** When salaries are equal, use ascending employee ID.

Unlike physical Spark partitions, window partitions represent **logical groups of rows**.

Spark may need to redistribute records to process each group correctly.

### 2. Assigning Row Numbers

```python
ranked = employees.withColumn(
    "rn", F.row_number().over(w)
)
```

`row_number()` assigns consecutive integers starting from 1 within each department.

For example:

| DEPARTMENT | FIRST_NAME | SALARY | rn |
|------------|------------|--------|----|
| IT | Shelley | 8900 | 1 |
| IT | Pat | 8100 | 2 |
| IT | John | 5500 | 3 |

To retrieve the two highest-paid employees per department:

```python
top2 = ranked.where(F.col("rn") <= 2)
```

### 3. GROUP BY vs. Window Functions

**GROUP BY:**

```python
employees.groupBy("DEPARTMENT") \
         .agg(F.max("SALARY"))
```

Produces one row per department.

**Window function:**

```python
employees.withColumn(
    "rn", F.row_number().over(w)
)
```

Preserves individual employee records while assigning a rank.

**Key distinction:** `GROUP BY` reduces the number of rows; window functions generally preserve the input rows until additional filtering is applied.

### 4. Physical Execution Plan

Our Spark 3.4 execution plan is:

```text
AdaptiveSparkPlan
+- Sort
   +- Exchange (range partitioning)
      +- Project
         +- Filter (rn <= 2)
            +- Window
               +- Sort
                  +- Exchange (hash partitioning)
                     +- Scan csv
```

Read the plan from bottom to top.

![Physical plan](figures/physical_plan_2.png)

**First Exchange — Hash Partitioning**

```text
hashpartitioning(DEPARTMENT, 200)
```

Brings records with the same department together.

**First Sort**

```text
DEPARTMENT ASC, SALARY DESC, ID ASC
```

Orders records so Spark can compute row numbers correctly.

**Window**

```text
row_number() OVER (
    PARTITION BY DEPARTMENT
    ORDER BY SALARY DESC, ID ASC
)
```

Computes the sequential ranking within each department.

**Filter**

```text
rn <= 2
```

Retains the two highest-ranked employees per department.

**Second Exchange — Range Partitioning**

```text
rangepartitioning(DEPARTMENT ASC, rn ASC, 200)
```

Prepares the output for global ordering.

**Final Sort**

```text
DEPARTMENT ASC, rn ASC
```

Produces the requested display order.

### 5. Why Are There Two Shuffles?

The two `Exchange` operators serve different purposes:

| Exchange | Purpose |
|----------|---------|
| Hash partitioning by DEPARTMENT | Group records for window computation |
| Range partitioning by DEPARTMENT and rn | Support global sorting of the final output |

**Important:** The final `orderBy()` can introduce an additional expensive shuffle even though the window already required sorting.

### 6. Experiment: Remove the Final ORDER BY

```python
top2_unsorted = (
    employees
    .withColumn("rn", F.row_number().over(w))
    .where(F.col("rn") <= 2)
    .select("DEPARTMENT", "FIRST_NAME", "SALARY", "rn")
)

top2_unsorted.explain("formatted")
```

Compare this with the original plan.

**Expected:** The final global `Sort` and range-partitioning `Exchange` disappear.

The window's own shuffle and sort remain necessary.

### 7. row_number() vs. rank() vs. dense_rank()

Three commonly used ranking functions are:

| Function | Behavior with ties |
|----------|--------------------|
| `row_number()` | Assigns a unique sequential number to every row |
| `rank()` | Equal values receive equal ranks; gaps may occur |
| `dense_rank()` | Equal values receive equal ranks; no gaps occur |

Example with salaries:

| Salary | row_number | rank | dense_rank |
|--------|------------|------|------------|
| 9000 | 1 | 1 | 1 |
| 8000 | 2 | 2 | 2 |
| 8000 | 3 | 2 | 2 |
| 7000 | 4 | 4 | 3 |

For tied values, `row_number()` needs an additional ordering attribute when deterministic results are required.


### Takeaways

- Window functions compute analytical values **without collapsing rows**.
- `partitionBy()` defines logical groups, not necessarily physical Spark partitions.
- `row_number()` is useful for top-k queries with a fixed number of results per group.
- Window computations frequently require shuffling and sorting.
- Final global ordering may require an additional shuffle.
- Deterministic tie-breaking is important for reproducible results.

### Discussion Questions

1. Why can't a simple `GROUP BY` return the top two employees per department?
2. Why does `row_number()` require records to be ordered within each department?
3. What is the difference between logical window partitions and physical Spark partitions?
4. Why are two `Exchange` operators present in the original physical plan?
5. What happens if the final `orderBy()` is removed?
6. When would `rank()` be preferable to `row_number()`?


In [27]:
%livy.pyspark
sql_top2 = spark.sql('''WITH ranked AS (
 SELECT DEPARTMENT, FIRST_NAME, SALARY, ID,
        ROW_NUMBER() OVER (PARTITION BY DEPARTMENT ORDER BY SALARY DESC, ID) AS rn
 FROM employees
)
SELECT DEPARTMENT, FIRST_NAME, SALARY, rn
FROM ranked WHERE rn <= 2 ORDER BY DEPARTMENT, rn''')
sql_top2.show(50, truncate=False)


## Example 5 — Window Functions with Spark SQL

The previous example demonstrated how to retrieve the top two employees per department using the **DataFrame API**.

We can express the exact same computation using **Spark SQL**, with a window function and a Common Table Expression (CTE).

### 1. SQL Window Functions

```sql
ROW_NUMBER() OVER (
    PARTITION BY DEPARTMENT
    ORDER BY SALARY DESC, ID ASC
) AS rn
```

This expression:

- Divides employees into groups based on `DEPARTMENT`.
- Sorts each group by salary in descending order.
- Uses employee ID as a deterministic tie-breaker.
- Assigns a sequential row number within each group.

It is equivalent to the DataFrame API:

```python
w = Window.partitionBy("DEPARTMENT") \
          .orderBy(F.desc("SALARY"), F.asc("ID"))

employees.withColumn("rn", F.row_number().over(w))
```

### 2. Common Table Expressions (CTEs)

A **CTE** defines a named intermediate query using the `WITH` clause:

```sql
WITH ranked AS (
    SELECT DEPARTMENT, FIRST_NAME, SALARY, ID,
           ROW_NUMBER() OVER (
               PARTITION BY DEPARTMENT
               ORDER BY SALARY DESC, ID
           ) AS rn
    FROM employees
)
SELECT DEPARTMENT, FIRST_NAME, SALARY, rn
FROM ranked
WHERE rn <= 2
ORDER BY DEPARTMENT, rn
```

The query consists of two logical steps:

**Step 1 — Compute rankings**

The CTE named `ranked` assigns a row number to each employee within their department.

**Step 2 — Filter rankings**

The outer query retains only employees with `rn <= 2`.

A CTE is useful because window-function results generally cannot be filtered directly in the `WHERE` clause of the same query block.

**Important:** CTEs are logical query constructs. Spark does not automatically materialize their results in memory or on disk.

### 3. DataFrame API vs. SQL

Both implementations perform the same relational computation:

| DataFrame API | Spark SQL |
|---------------|-----------|
| `Window.partitionBy()` | `OVER (PARTITION BY ...)` |
| `Window.orderBy()` | `ORDER BY` inside `OVER` |
| `F.row_number().over(w)` | `ROW_NUMBER() OVER (...)` |
| `withColumn("rn", ...)` | `... AS rn` |
| `where(F.col("rn") <= 2)` | `WHERE rn <= 2` |
| `orderBy(...)` | Final `ORDER BY` |

Both are processed by the **Catalyst optimizer**.

### 4. Compare the Physical Plans

```python
print("DataFrame plan:")
top2.explain(mode="formatted")

print("SQL plan:")
sql_top2.explain(mode="formatted")
```

The two queries should produce equivalent physical execution plans, containing:

1. `Scan` — Read the employee records.
2. `Exchange` — Redistribute records by department.
3. `Sort` — Order employees within each department.
4. `Window` — Compute row numbers.
5. `Filter` — Retain rows with `rn <= 2`.
6. `Project` — Select the required columns.
7. `Exchange` and `Sort` — Produce globally ordered results.

Internal expression identifiers may differ, but the underlying execution strategy should be equivalent.

### 5. Why Does This Matter?

Spark SQL is **not a separate execution engine** from the DataFrame API.

Both interfaces are translated into logical plans, optimized by Catalyst, and eventually executed using Spark's physical operators.

Choosing between SQL and DataFrames is therefore primarily a matter of expressiveness, readability and integration with application code—not an inherent performance advantage of one interface.

### Takeaways

- Spark SQL supports analytical window functions.
- CTEs help structure multi-step queries without automatically materializing intermediate results.
- Filtering on computed window values typically requires a subquery or CTE.
- SQL and DataFrame implementations can produce equivalent physical plans.
- Window functions are essential for top-k queries within groups.

### Discussion Questions

1. Why can't we filter on `rn` in the same query block that computes `ROW_NUMBER()`?
2. Does using a CTE necessarily introduce an additional Spark job?
3. Would replacing the CTE with a nested subquery change the logical result?
4. If SQL and DataFrame versions produce equivalent physical plans, should one necessarily be faster?


In [29]:
%livy.pyspark
print("DataFrame plan:")
top2.explain(mode="formatted")

print("SQL plan:")
sql_top2.explain(mode="formatted")

## 6. Data cleaning: strings, dates, conditional buckets (10 min)
Teach `when`, `length`, `split`, `explode`, date extraction, and explicit casting. For the project, text length is in **characters**, not bytes; multi-valued categories must be expanded before grouping.


In [31]:
%livy.pyspark
cleaned = (employees
    .withColumn('hire_day', F.to_date(F.substring('HIRE_DATE', 1, 10)))
    .withColumn('hire_year', F.year('hire_day'))
    .withColumn('salary_band', F.when(F.col('SALARY') < 3000, 'low')
        .when(F.col('SALARY') < 7000, 'mid').otherwise('high'))
    .withColumn('email_domain', F.element_at(F.split('EMAIL', '@'), -1)))
cleaned.groupBy('hire_year', 'salary_band').count().orderBy('hire_year', 'salary_band').show(30)
spark.createDataFrame([('a', 'Food, Restaurants'), ('b', 'Cafes, Food')],
                      ['id','categories']).select('id', F.explode(F.split('categories', ',\s*')).alias('category')).show()


## Example 6 — Data Cleaning, Feature Engineering & Exploding Arrays

Real-world datasets frequently contain dates stored as strings, missing values, categorical attributes, and fields containing multiple values.

Before performing analytical queries, these attributes often need to be **cleaned, transformed, or normalized**.

This example introduces common PySpark functions used for data preparation.

### 1. Extracting Dates from Strings

The original `HIRE_DATE` column contains strings such as:

```text
2022-01-01T19:34:33.000+02:00
```

We can extract the date component:

```python
F.to_date(F.substring("HIRE_DATE", 1, 10))
```

The operations are:

1. `substring()` extracts the first 10 characters (`2022-01-01`).
2. `to_date()` converts the string to a Spark `DateType`.
3. `year()` extracts the calendar year from the resulting date.

Example:

```python
employees.withColumn(
    "hire_day",
    F.to_date(F.substring("HIRE_DATE", 1, 10))
).withColumn(
    "hire_year",
    F.year("hire_day")
)
```

**Why use a proper date type?**

Date types allow Spark to perform calendar-aware operations, comparisons, grouping, and date arithmetic.

Note that extracting the first 10 characters preserves the date written in the source string; it does not convert the timestamp to another timezone.

### 2. Deriving Categorical Attributes

Conditional expressions can transform numeric values into categories.

```python
F.when(F.col("SALARY") < 3000, "low") \
 .when(F.col("SALARY") < 7000, "mid") \
 .otherwise("high")
```

The resulting salary bands are:

| Salary | Band |
|--------|------|
| Below 3000 | low |
| 3000–6999 | mid |
| 7000 and above | high |

Spark evaluates the conditions in order, similarly to SQL `CASE WHEN`.

**Important:** Our current expression classifies NULL salaries as `high`, because neither comparison evaluates to true. A production pipeline should handle missing values explicitly:

```python
F.when(F.col("SALARY").isNull(), "unknown") \
 .when(F.col("SALARY") < 3000, "low") \
 .when(F.col("SALARY") < 7000, "mid") \
 .otherwise("high")
```

### 3. String Splitting and Array Operations

Consider:

```python
F.element_at(F.split("EMAIL", "@"), -1)
```

`split()` transforms a string into an array:

```text
"student@example.com"
         ↓
["student", "example.com"]
```

`element_at(..., -1)` extracts the last element:

```text
"example.com"
```

In Spark:

- `split()` returns an array of strings.
- `element_at()` uses 1-based indexing.
- Negative indices count backward from the end.

These operations are useful for processing delimited fields.

### 4. Combining Transformations

```python
cleaned = (
    employees
    .withColumn("hire_day",
        F.to_date(F.substring("HIRE_DATE", 1, 10)))
    .withColumn("hire_year", F.year("hire_day"))
    .withColumn("salary_band",
        F.when(F.col("SALARY") < 3000, "low")
         .when(F.col("SALARY") < 7000, "mid")
         .otherwise("high"))
    .withColumn("email_domain",
        F.element_at(F.split("EMAIL", "@"), -1))
)
```

Each `withColumn()` adds or replaces an attribute.

The transformations are lazy, and Spark can optimize the complete expression pipeline.

After deriving the new attributes, we can aggregate:

```python
cleaned.groupBy("hire_year", "salary_band") \
       .count() \
       .orderBy("hire_year", "salary_band") \
       .show()
```

This calculates the number of employees in each combination of hiring year and salary category.

### 5. Exploding Multi-Valued Attributes

Some datasets store multiple values in one field:

```text
id | categories
---+------------------
a  | Food, Restaurants
b  | Cafes, Food
```

Using:

```python
F.split("categories", r",\s*")
```

we obtain arrays:

```text
a → ["Food", "Restaurants"]
b → ["Cafes", "Food"]
```

The `explode()` function transforms each array element into a separate row:

```python
categories_df.select(
    "id",
    F.explode(
        F.split("categories", r",\s*")
    ).alias("category")
)
```

Result:

```text
id | category
---+------------
a  | Food
a  | Restaurants
b  | Cafes
b  | Food
```

**Key distinction:** Unlike ordinary column transformations, `explode()` can increase the number of rows.

Each input record contributes one output record per array element.

### 6. Why This Matters for the Semester Assignment

The Yelp dataset stores business categories as comma-separated strings.

A business can belong to multiple categories, such as:

```text
Restaurants, Italian, Pizza
```

By splitting and exploding the category list, we can associate that business with each category independently.

When reviews are subsequently joined to these categories, **each review contributes to every category assigned to its business**.

Consequently, the sum of category-level review counts may exceed the number of distinct reviews.

This is expected and must be considered when interpreting aggregation results.

### 7. Distributed Execution

Most operations introduced here are **row-local transformations**:

- `substring()`
- `to_date()`
- `year()`
- `when()`
- `split()`
- `element_at()`
- `explode()`

They generally require no shuffle.

However:

```python
cleaned.groupBy("hire_year", "salary_band").count()
```

introduces a grouped aggregation that may require a shuffle.

Likewise, the final `orderBy()` may require another data redistribution to establish global ordering.

### Experiment: Inspect the Plan

```python
cleaned.select(
    "FIRST_NAME", "hire_year", "salary_band"
).explain("formatted")

cleaned.groupBy(
    "hire_year", "salary_band"
).count().explain("formatted")
```

Compare the two physical plans.

Look for:

- `Project` — Derived columns and expressions.
- `HashAggregate` — Grouped aggregation.
- `Exchange` — Data redistribution.

You can also inspect the plan for `explode()`:

```python
categories_df.select(
    "id",
    F.explode(F.split("categories", r",\s*")).alias("category")
).explain("formatted")
```

Look for the `Generate` operator, which implements row-generating expressions such as `explode()`.

### Takeaways

- Data cleaning converts raw values into structured, analysis-ready attributes.
- Date and string functions are fundamental tools for preparing real-world datasets.
- Conditional expressions implement SQL-style classification logic.
- `split()` creates arrays from delimited strings.
- `explode()` converts multi-valued attributes into multiple rows.
- Row-local transformations generally avoid shuffles.
- Aggregations and global ordering can introduce shuffle boundaries.
- Multi-valued categorical attributes require careful interpretation of subsequent counts.

### Discussion Questions

1. Why should date strings be converted into proper date types?
2. What happens if a salary is NULL in the original classification expression?
3. What is the difference between `split()` and `explode()`?
4. Why can `explode()` increase the total number of rows?
5. Why might the sum of review counts across Yelp categories exceed the total number of unique reviews?
6. Which operations in this example require a shuffle, and why?


## 7. Shuffle and physical plans: read what Spark actually executes (15 min)
`Exchange` usually indicates repartitioning/data movement. Look for `HashAggregate`, `Sort`, `Window`, and the join implementation. **An Exchange does not necessarily imply bytes traveled between physical machines**; local shuffle is possible. Compare narrow filter to wide aggregation.


In [34]:
%livy.pyspark
print('Filter / projection:')
employees.filter(F.col('SALARY') > 5000).select('ID','SALARY').explain(mode='formatted')
print('Group by:')
employees.groupBy('DEPARTMENT').agg(F.avg('SALARY')).explain(mode='formatted')
print('Repartition then group by:')
employees.repartition(4, 'DEPARTMENT').groupBy('DEPARTMENT').count().explain(mode='formatted')


## Example 7 — Shuffles, Physical Plans & Repartitioning

This example compares three DataFrame queries to understand **when Spark redistributes data between partitions** and how this appears in physical execution plans.

A **shuffle** occurs when Spark redistributes data across partitions to satisfy the requirements of an operation.

Shuffles can be expensive because they may involve network communication, serialization, disk I/O, and sorting or aggregation.

### 1. Filter and Projection — No Shuffle

```python
employees.filter(F.col("SALARY") > 5000) \
         .select("ID", "SALARY") \
         .explain(mode="formatted")
```

The query performs:

- **Filter:** Retain employees earning more than 5,000.
- **Projection:** Keep only ID and SALARY.

These are narrow transformations: each record can be processed independently within its existing partition.

The physical plan should contain operators such as:

```text
Project / Filter
   +- Scan csv
```

Catalyst may combine or eliminate projection operators when column pruning already produces the required output.

**Observation:** No `Exchange` is needed because filtering and projection do not require communication between partitions.

### 2. Grouped Aggregation — Shuffle Required

```python
employees.groupBy("DEPARTMENT") \
         .agg(F.avg("SALARY")) \
         .explain(mode="formatted")
```

The query calculates the average salary per department.

A typical physical plan is:

```text
HashAggregate (final)
   +- Exchange (hash partitioning)
      +- HashAggregate (partial)
         +- Scan csv
```

Spark performs:

1. **Partial aggregation:** Compute intermediate sums and counts locally.
2. **Exchange:** Redistribute intermediate results by department.
3. **Final aggregation:** Combine intermediate results for each department.

The Exchange typically contains:

```text
hashpartitioning(DEPARTMENT, 200)
ENSURE_REQUIREMENTS
```

`ENSURE_REQUIREMENTS` indicates that Spark inserted the exchange to satisfy the aggregation's distribution requirements.

### 3. Explicit Repartitioning Before Aggregation

```python
employees.repartition(4, "DEPARTMENT") \
         .groupBy("DEPARTMENT") \
         .count() \
         .explain(mode="formatted")
```

This query explicitly partitions the input by department before grouping.

Our physical plan contains:

```text
HashAggregate (final)
   +- HashAggregate (partial)
      +- Exchange (hash partitioning)
         +- Scan csv
```

Notice that there is only **one Exchange**, despite performing both `repartition()` and `groupBy()`.

Why?

`repartition(4, "DEPARTMENT")` already ensures that records with the same department are assigned to the same partition.

The subsequent grouped aggregation can exploit this arrangement without introducing another shuffle.

The physical plan contains:

```text
hashpartitioning(DEPARTMENT, 4)
REPARTITION_BY_NUM
```

Unlike `ENSURE_REQUIREMENTS`, the `REPARTITION_BY_NUM` origin indicates an explicitly requested repartitioning.

### 4. Partitioning Requirements and Optimization

Spark operators have requirements concerning how their input data must be distributed.

For example:

- `Filter` and `Project` generally have no special distribution requirements.
- Grouped aggregation needs records with the same grouping key to be colocated.
- Sort-Merge Join typically needs both inputs clustered and sorted by their join keys.
- Window functions generally require appropriate grouping and ordering.

**Important:** If an existing partitioning arrangement already satisfies an operator's requirements, Spark may avoid introducing an additional Exchange.

This is an example of optimization based on **physical data properties**.

### 5. Does Repartitioning Always Improve Performance?

No.

Explicit repartitioning introduces a shuffle, which can be expensive.

For our example, the original dataset contains only 19 employees. Redistributing such a small dataset provides no practical performance benefit.

Repartitioning can help when:

- Increasing parallelism for a large workload.
- Preparing data for repeated operations on the same key.
- Correcting poor partition distribution.
- Controlling partition counts for output or downstream processing.

However, unnecessary repartitioning can increase execution time and network traffic.

### 6. Partition Count vs. Data Distribution

```python
employees.repartition(4, "DEPARTMENT")
```

Requests four hash partitions.

However:

- Four partitions do not imply four executors.
- Four partitions do not guarantee balanced workloads.
- Rows with the same department are assigned to the same partition.
- Different departments may hash to the same partition.
- Some partitions may be empty.

With only six departments, the resulting distribution is unlikely to represent a balanced parallel workload.

### 7. Experiment: Compare Different Partitioning Keys

Compare:

```python
# Partition by the aggregation key
employees.repartition(4, "DEPARTMENT") \
         .groupBy("DEPARTMENT").count() \
         .explain("formatted")

# Partition by a different key
employees.repartition(4, "ID") \
         .groupBy("DEPARTMENT").count() \
         .explain("formatted")
```

**Question:** Why does the second query generally require an additional Exchange?

Because partitioning by `ID` does not guarantee that employees from the same department are colocated.

### 8. Experiment: Inspect the Partition Distribution

```python
partitioned = employees.repartition(4, "DEPARTMENT")

partitioned.withColumn(
    "partition_id",
    F.spark_partition_id()
).groupBy(
    "partition_id"
).count().orderBy("partition_id").show()
```

This displays the number of records assigned to each nonempty partition.

Notice that the distribution may be uneven and empty partitions do not appear in the grouped result.

### Takeaways

- Narrow transformations generally avoid shuffles.
- Grouped aggregations typically require compatible key partitioning.
- An `Exchange` operator indicates data redistribution.
- `ENSURE_REQUIREMENTS` identifies an exchange inserted to satisfy execution requirements.
- `REPARTITION_BY_NUM` identifies an explicit repartitioning request.
- Existing partitioning can sometimes be reused to avoid additional shuffles.
- Repartitioning does not guarantee balanced workloads or faster execution.
- Physical execution plans help explain how Spark organizes distributed computations.

### Discussion Questions

1. Why do filtering and projection not require a shuffle?
2. Why does grouped aggregation require records with the same key to be colocated?
3. Why does repartitioning by `DEPARTMENT` avoid a second Exchange?
4. What happens when we repartition by `ID` but group by `DEPARTMENT`?
5. Does increasing the number of partitions always improve performance?
6. What is the difference between `REPARTITION_BY_NUM` and `ENSURE_REQUIREMENTS`?


## 8. Parquet and partition pruning (15 min)
The project first converts raw sources into Parquet and partitions reviews by year. Here we partition employees by **hire year** to illustrate `PartitionFilters`, `ReadSchema` and column projection. `DEMO_PARQUET_PATH` must be a writable cluster-visible path. This paragraph writes data; run it **once**.


In [37]:
%livy.pyspark
DEMO_PARQUET_PATH = 'hdfs://nchalv-1:54310/user/ubuntu/ta/advanced-db/spark-examples/output'  # EDIT: writable cluster-visible path
(cleaned.drop('HIRE_DATE')
    .write.mode('overwrite').partitionBy('hire_year').parquet(DEMO_PARQUET_PATH))
pq = spark.read.parquet(DEMO_PARQUET_PATH)
filtered_pq = pq.where(F.col('hire_year') == 2020).select('ID','DEPARTMENT','SALARY')
filtered_pq.show(20)
filtered_pq.explain(mode='formatted')


## Example 8 — Parquet, Partition Pruning & Columnar Storage

Apache Spark supports **Parquet**, a column-oriented storage format designed for efficient analytical processing.

Unlike CSV and JSON, Parquet stores typed, compressed column data and supports optimizations such as **column pruning**, predicate pushdown and vectorized reading.

This example demonstrates how organizing a dataset by partition columns can significantly reduce the amount of data Spark needs to read.

### 1. Writing Partitioned Parquet

```python
(cleaned.drop("HIRE_DATE")
    .write.mode("overwrite")
    .partitionBy("hire_year")
    .parquet(DEMO_PARQUET_PATH))
```

`partitionBy("hire_year")` organizes the output into directories based on the year.

The resulting structure resembles:

```text
output/
├── hire_year=2018/
│   └── part-....parquet
├── hire_year=2020/
│   └── part-....parquet
├── hire_year=2021/
│   └── part-....parquet
└── hire_year=2022/
    └── part-....parquet
```

Each directory contains the records for the corresponding year.

**Important distinction:**

- `repartition()` controls the distribution of records during execution.
- `write.partitionBy()` organizes records into partition directories when writing a dataset.

These are related but different mechanisms.

### 2. Partition Pruning

After reading the Parquet dataset:

```python
pq = spark.read.parquet(DEMO_PARQUET_PATH)

filtered_pq = pq.where(
    F.col("hire_year") == 2020
).select("ID", "DEPARTMENT", "SALARY")
```

The physical plan contains:

```text
PartitionFilters: [
    isnotnull(hire_year),
    (hire_year = 2020)
]
```

Spark recognizes that `hire_year` is a partition column.

It can therefore **skip entire directories** containing records from other years.

Instead of scanning all partitions, Spark only needs to read:

```text
hire_year=2020/
```

This optimization is called **partition pruning**.

The improvement can be substantial when processing large datasets distributed across many partition directories.

### 3. Column Pruning

The physical plan contains:

```text
ReadSchema: struct<ID:int,SALARY:int,DEPARTMENT:string>
```

Only three data columns are required by the query.

Spark therefore avoids reading the remaining column data from the Parquet files.

Notice that `hire_year` is not included in `ReadSchema`.

This is because it is a partition column: Spark can reconstruct its value from the directory name.

**CSV vs. Parquet:**

- CSV is row-oriented text. Spark generally reads text records even when only a few columns are needed.
- Parquet stores column data separately, allowing Spark to skip unnecessary column chunks.

This is one of the principal reasons Parquet is well suited for analytical workloads.

### 4. Vectorized Parquet Reading

Our execution plan contains:

```text
Batched: true
```

This indicates that Spark uses a vectorized Parquet reader.

The reader processes column values in batches rather than decoding each record individually.

The plan also contains:

```text
ColumnarToRow
```

This operator converts columnar batches into Spark's internal row representation for downstream processing.

**Important:** `ColumnarToRow` does not negate the benefits of vectorized Parquet reading.

### 5. Understanding the Physical Plan

Our query produces:

```text
Project
   +- ColumnarToRow
      +- Scan parquet
```

The scan includes:

```text
PartitionFilters: [hire_year = 2020]
ReadSchema: struct<ID:int,SALARY:int,DEPARTMENT:string>
Batched: true
```

The physical plan demonstrates three mechanisms:

| Mechanism | Effect |
|-----------|--------|
| Partition pruning | Skips irrelevant partition directories |
| Column pruning | Avoids reading unnecessary Parquet columns |
| Vectorized reading | Processes column data in batches |

Notice that there is **no Exchange operator**.

The query requires no shuffle because filtering and projection can be processed independently on the relevant input partitions.

### 6. Partition Pruning vs. Predicate Pushdown

These are related but distinct optimizations.

**Partition pruning**

Uses partition-column values to eliminate entire directories or files before scanning their records.

Example:

```python
pq.where(F.col("hire_year") == 2020)
```

**Predicate pushdown**

Pushes supported filter conditions closer to the storage reader.

For Parquet, this can enable skipping data using metadata such as row-group statistics.

Example:

```python
pq.where(F.col("SALARY") > 5000)
```

The physical plan may include:

```text
PushedFilters: [IsNotNull(SALARY), GreaterThan(SALARY,5000)]
```

A pushed filter does not guarantee that every nonmatching row is eliminated without being decoded.

### 7. Experiment: Compare Different Filters

**A. Filter on the partition column**

```python
pq.where(F.col("hire_year") == 2020) \
  .explain("formatted")
```

Look for `PartitionFilters`.

**B. Filter on an ordinary data column**

```python
pq.where(F.col("SALARY") > 5000) \
  .explain("formatted")
```

Look for `PushedFilters`.

**C. Combine both filters**

```python
pq.where(
    (F.col("hire_year") == 2020) &
    (F.col("SALARY") > 5000)
).select("ID", "SALARY").explain("formatted")
```

Observe how partition pruning, predicate pushdown and column pruning can work together.

### 8. Experiment: Inspect HDFS

Use the HDFS command line to inspect the output:

```bash
hdfs dfs -ls -R /user/ubuntu/ta/advanced-db/spark-examples/output
```

Identify:

- The partition directories.
- The Parquet data files.
- The number of files generated for each year.
- The presence of output metadata or marker files.


### Takeaways

- Parquet is a **column-oriented, typed and compressed** storage format.
- `write.partitionBy()` organizes output files into directories based on partition-column values.
- **Partition pruning** avoids scanning irrelevant directories.
- **Column pruning** avoids reading unnecessary Parquet columns.
- **Predicate pushdown** allows supported filters to be evaluated closer to the data source.
- `Batched: true` indicates vectorized reading.
- `ColumnarToRow` converts vectorized batches for row-oriented operators.
- Filtering and projection do not inherently require shuffles.
- Physical plans provide evidence that storage-level optimizations are being applied.

### Discussion Questions

1. What is the difference between `repartition()` and `write.partitionBy()`?
2. Why is `hire_year` missing from the Parquet `ReadSchema`?
3. How does partition pruning reduce disk I/O?
4. Why can Parquet skip unnecessary columns more efficiently than CSV?
5. What is the difference between `PartitionFilters` and `PushedFilters`?
6. What does `Batched: true` indicate?
7. Would partitioning a dataset by a column containing millions of distinct values necessarily be a good idea?


In [39]:
%livy.pyspark
pq.where(F.col("hire_year") == 2020) \
  .explain("formatted")
  


In [40]:
%livy.pyspark
pq.where(F.col("SALARY") > 5000) \
  .explain("formatted")


In [41]:
%livy.pyspark
pq.where(
    (F.col("hire_year") == 2020) &
    (F.col("SALARY") > 5000)
).select("ID", "SALARY").explain("formatted")

## 9. Caching, recomputation, and measurement (10 min)
Cache benefits repeat usage when recomputation costs more than storage. `cache()` is lazy: an action materializes it. Use the **Storage** tab to check persistence. On tiny employee data, timing differences are mostly overhead—**do not make performance claims from this dataset**.


In [43]:
%livy.pyspark
import time

base = (
    employees
    .select("DEPARTMENT", "SALARY")
    .where(F.col("SALARY").isNotNull())
)

base.cache()

print("Cached before action:", base.is_cached)

t0 = time.perf_counter()
print("Rows:", base.count())
print("Materialization time:", round(time.perf_counter() - t0, 3), "s")

print("Inspect the Storage tab NOW.")

In [44]:
%livy.pyspark
print("Average salary by department:")
base.groupBy("DEPARTMENT").agg(
    F.avg("SALARY").alias("avg_salary")
).show()

print("Maximum salary by department:")
base.groupBy("DEPARTMENT").agg(
    F.max("SALARY").alias("max_salary")
).show()

print("Physical plan:")
base.groupBy("DEPARTMENT").agg(
    F.avg("SALARY")
).explain(mode="formatted")

In [45]:
%livy.pyspark
base.unpersist(blocking=True)

print("Still cached:", base.is_cached)
print("Cache released. Refresh the Storage tab.")

## Example 9 — Caching, Recomputation & Performance Measurement

Spark uses **lazy evaluation** and records the lineage of transformations required to compute a DataFrame.

By default, repeatedly executing actions on the same DataFrame may cause Spark to recompute its transformations and reread its input data.

**Caching** allows Spark to retain intermediate results for reuse.

### 1. Creating a Cache

```python
base = employees.select("DEPARTMENT", "SALARY") \
                .where(F.col("SALARY").isNotNull())

base.cache()
base.count()
```

- `cache()` marks the DataFrame for persistence.
- It does not immediately materialize the data.
- `count()` triggers an action that populates the cache.

The cached representation can then be reused by subsequent queries.

**Important:** Keep the DataFrame cached while inspecting Spark UI.

### 2. Reusing Cached Data

```python
base.groupBy("DEPARTMENT").agg(F.avg("SALARY")).collect()
base.groupBy("DEPARTMENT").agg(F.max("SALARY")).collect()
```

Both queries reuse the same filtered, projected DataFrame.

Without caching, Spark may reread and reprocess the original CSV for each query.

With caching, Spark can access the previously materialized data.

Inspect the physical plan:

```python
base.groupBy("DEPARTMENT").agg(
    F.avg("SALARY")
).explain("formatted")
```

Look for:

`InMemoryTableScan`

This indicates that the query accesses cached data instead of directly scanning the original CSV.

### 3. Spark UI — Storage Tab

After materializing the cache, open:

**Spark UI → Storage**

The Storage tab can display:

- Cached RDDs and DataFrames.
- Storage levels.
- Number of cached partitions.
- Memory consumption.
- Disk usage, where applicable.

**Do not call `unpersist()` before inspecting this tab!**

Once cached blocks have been removed, their entries disappear from Storage.

### 4. Releasing Cached Data

```python
base.unpersist(blocking=True)
```

This releases the cached blocks.

The next action requiring those records may need to recompute the DataFrame from its original lineage.

`blocking=True` waits for the unpersist operation to complete.

Caching does not permanently store the dataset.

### 5. When Is Caching Useful?

Caching is particularly beneficial when:

- The same DataFrame is reused by multiple queries.
- Preparing the DataFrame is computationally expensive.
- Repeated scans or transformations dominate execution time.
- Sufficient executor memory is available.

Caching may be counterproductive when:

- A DataFrame is used only once.
- The dataset is inexpensive to recompute.
- The cached data consumes too much memory.
- Cache materialization and memory pressure outweigh reuse benefits.

### 6. Measuring Performance

Use wall-clock measurements around actions that execute the desired computation.

```python
import time

t0 = time.perf_counter()
result = base.groupBy("DEPARTMENT").agg(F.avg("SALARY")).collect()
elapsed = time.perf_counter() - t0

print("Elapsed:", round(elapsed, 3), "seconds")
```

For meaningful comparisons:

1. Compare equivalent computations.
2. Separate cache population from cache reuse.
3. Repeat experiments multiple times.
4. Report the median rather than relying on a single run.
5. Inspect Spark UI for task execution time, shuffle activity and caching behavior.

**Caution:** Our employee dataset contains only 19 rows. Performance differences are dominated by overhead and should not be used to draw general conclusions about caching efficiency.

### 7. Experiment: Cache Lifecycle

Run these steps separately.

**A. Define and materialize the cache**

```python
base.cache()
base.count()
print("Cached:", base.is_cached)
```

Inspect the Storage tab.

**B. Execute a query using cached data**

```python
base.groupBy("DEPARTMENT").agg(
    F.avg("SALARY")
).explain("formatted")
```

Look for `InMemoryTableScan`.

**C. Remove the cache**

```python
base.unpersist(blocking=True)
print("Cached:", base.is_cached)
```

Refresh the Storage tab and observe what changed.

### Takeaways

- Spark normally uses lineage to recompute data when required.
- `cache()` is lazy; an action materializes the cached data.
- Cached results can be reused across multiple queries.
- `InMemoryTableScan` indicates cached-data access.
- The Storage tab displays persisted datasets in the running application.
- `unpersist()` releases cached data.
- Caching improves performance only when the benefits of reuse outweigh its costs.

### Discussion Questions

1. Why does `cache()` alone not populate the Storage tab?
2. Why does the Storage tab become empty after `unpersist()`?
3. What is the difference between caching and writing Parquet?
4. Why might caching a 19-row DataFrame provide no measurable speedup?
5. What happens if a cached partition is evicted from memory?
6. Why is caching a large text column potentially wasteful when subsequent queries require only numeric attributes?


## 10. Spark UI inspection challenge (10 min)
Run an action on `top2` and inspect **Jobs → Stages → Tasks** in the Spark application UI (often port 4040, or via Livy/session UI). Explain: number of input partitions; tasks per stage; shuffle read/write; spill; SQL physical plan. For this tiny input, metrics may be near zero. Larger synthetic data is demonstrated below.


In [48]:
%livy.pyspark
# Bounded synthetic data; no extra file download. Change n only if lab capacity allows.
n = 100000
synthetic = spark.range(n).repartition(8).select(
    (F.col('id') % 50).alias('key'), (F.col('id') % 1000).alias('value'))
result = synthetic.groupBy('key').agg(F.sum('value').alias('total')).orderBy(F.desc('total'))
result.collect()  # real action
result.explain(mode='formatted')


## Example 10 — Spark UI, Execution Stages & Adaptive Query Execution

This example generates a synthetic dataset with 100,000 records, redistributes it across 8 partitions, groups records by key, and globally sorts the aggregated results.

Its purpose is to connect **DataFrame transformations, physical operators, shuffle boundaries, stages, tasks, and Adaptive Query Execution (AQE)**.

### 1. Generating Synthetic Data

```python
n = 100000

synthetic = (
    spark.range(n)
    .repartition(8)
    .select(
        (F.col("id") % 50).alias("key"),
        (F.col("id") % 1000).alias("value")
    )
)
```

The dataset contains:

- 100,000 records.
- 50 distinct keys (`0` through `49`).
- Numeric values between `0` and `999`.
- 8 partitions after explicit repartitioning.

`spark.range()` generates records in parallel without requiring an external input file.

In our execution, the initial `Range` operator uses **10 input splits**.

The explicit `repartition(8)` redistributes the data into eight partitions.

### 2. Aggregation and Global Sorting

```python
result = (
    synthetic
    .groupBy("key")
    .agg(F.sum("value").alias("total"))
    .orderBy(F.desc("total"))
)

result.collect()
result.explain(mode="formatted")
```

The query:

1. Groups records by `key`.
2. Computes the sum of `value` for each key.
3. Orders the 50 resulting groups by descending total.

Calling `collect()` executes the query and returns the small final result to the driver.

### 3. Understanding the Three Exchanges

The initial physical plan contains:

```text
Sort
 +- Exchange (range partitioning)
    +- HashAggregate (final)
       +- Exchange (hash partitioning)
          +- HashAggregate (partial)
             +- Project
                +- Exchange (round robin)
                   +- Range
```

Schematically:

![Execution Pipeline]()


There are three shuffle operations.

| Exchange | Partitioning | Purpose |
|----------|--------------|---------|
| 1 | Round robin, 8 partitions | Explicit repartitioning |
| 2 | Hash by key, 200 partitions | Group partial aggregates |
| 3 | Range by total, 200 partitions | Support global sorting |

The first shuffle is requested by the programmer.

The other two are inserted by Spark to satisfy operator distribution requirements.

### 4. Partial Aggregation Reduces Communication

Our final physical plan reports:

| Stage output | Records | Estimated data size |
|--------------|---------|---------------------|
| After repartitioning | 100,000 | 1,562.5 KiB |
| After partial aggregation | 400 | 9.4 KiB |
| After final aggregation | 50 | 1,200 B |

Why does partial aggregation produce 400 records?

Each of the eight partitions contains records belonging to all 50 keys.

Each partition computes one partial sum per key.

Therefore:

`8 partitions × 50 keys = 400 partial aggregates`

After the shuffle, Spark combines partial sums belonging to the same key.

The final result contains only **50 records**.

**Key observation:** Partial aggregation can dramatically reduce the amount of data that needs to be shuffled.

### 5. Adaptive Query Execution (AQE)

The physical plan contains:

```text
AdaptiveSparkPlan
Arguments: isFinalPlan=true
```

It also displays:

```text
== Initial Plan ==
...
== Final Plan ==
...
```

The initial plan is generated before execution.

The final plan reflects optimizations made using runtime statistics.

Our final plan contains:

```text
AQEShuffleRead
Arguments: coalesced
```

This indicates that Spark has combined small shuffle partitions to reduce scheduling overhead.

For example, Spark initially requests 200 shuffle partitions for grouped aggregation and global sorting.

After observing the small shuffle outputs, AQE can reduce the number of partitions consumed by downstream stages.

**Important:** AQE does not eliminate the required shuffles in this example. It adapts how their outputs are processed.

### 6. Spark UI — Jobs, Stages & Tasks

Open the **Spark Application UI**, usually available through port 4040 or through the YARN tracking interface.

Inspect the following sections.

**SQL / DataFrame**

- Find the executed query.
- Identify the physical operators.
- Inspect the shuffle and aggregation metrics.
- Compare initial and adaptive execution behavior.

**Jobs**

- Identify jobs associated with the action.
- Examine their completion times.
- Follow links to individual stages.

**Stages**

- Count the tasks in each stage.
- Identify shuffle read and shuffle write metrics.
- Examine task execution times and data distribution.

**Executors**

- Inspect active executors.
- Observe completed tasks, memory usage and shuffle statistics.

### 7. Understanding ShuffleQueryStage

AQE divides execution into query stages whose shuffle outputs can be materialized and measured.

Our plan contains:

```text
ShuffleQueryStage
Statistics(sizeInBytes=9.4 KiB, rowCount=400)
```

These statistics describe the output of an executed shuffle query stage.

AQE can use this information to modify the remaining execution plan.

This is different from estimates available to Catalyst before execution.

### 8. Experiment: Disable AQE

Compare the query with and without Adaptive Query Execution.

```python
original_aqe = spark.conf.get("spark.sql.adaptive.enabled")

spark.conf.set("spark.sql.adaptive.enabled", "false")

result_no_aqe = (
    synthetic
    .groupBy("key")
    .agg(F.sum("value").alias("total"))
    .orderBy(F.desc("total"))
)

result_no_aqe.collect()
result_no_aqe.explain("formatted")

spark.conf.set("spark.sql.adaptive.enabled", original_aqe)
```

Compare:

- Physical plans.
- Number of shuffle tasks.
- Number of stages.
- Shuffle read/write statistics.
- Execution times.

Without AQE, Spark uses its non-adaptive physical plan, generally retaining the configured shuffle partition count unless another optimization changes it.

### 9. Experiment: Remove Explicit Repartitioning

Try:

```python
without_repartition = (
    spark.range(n)
    .select(
        (F.col("id") % 50).alias("key"),
        (F.col("id") % 1000).alias("value")
    )
    .groupBy("key")
    .agg(F.sum("value").alias("total"))
    .orderBy(F.desc("total"))
)

without_repartition.explain("formatted")
```

**Question:** Why does one `Exchange` disappear?

The initial round-robin shuffle is no longer required because `repartition(8)` has been removed.

The aggregation and global sorting may still require shuffles.

### 10. Spark UI Inspection Challenge

Using the Spark UI, answer the following:

1. How many jobs were triggered by the action?
2. How many tasks processed the initial `Range`?
3. How many output partitions were created by `repartition(8)`?
4. How many shuffle partitions did Spark initially plan?
5. How many tasks processed the shuffle outputs after AQE optimization?
6. How many bytes were written and read during the shuffle?
7. Did any task spill data to memory or disk?
8. What changed when AQE was disabled?
9. What changed when explicit repartitioning was removed?

Do not infer all these answers from the DataFrame code alone. Inspect the actual execution metrics.

### Takeaways

- Spark translates transformations into a physical execution plan.
- `Exchange` operators commonly indicate shuffle boundaries.
- Shuffles separate stages of distributed computation.
- Partial aggregation reduces the amount of data exchanged.
- `spark.sql.shuffle.partitions` defines the initial shuffle partition count.
- AQE uses runtime statistics to adapt execution.
- `AQEShuffleRead coalesced` indicates adaptive partition coalescing.
- Spark UI reveals the actual jobs, stages, tasks and runtime metrics.
- A physical plan and its actual execution metrics provide complementary information.

### Discussion Questions

1. Why does the query contain three shuffle exchanges?
2. Why are there only 400 partial aggregates from 100,000 input records?
3. What is the difference between the initial and final adaptive plans?
4. Why can 200 configured shuffle partitions result in fewer downstream tasks?
5. Why doesn't AQE simply eliminate every shuffle?
6. How can examining Spark UI help identify inefficient data distribution?


---
# Optional self-study / semester-project bridge
The following mini-experiments illustrate nontrivial features without using or revealing Yelp/ACS/GloVe answers. Run only on Spark **4.2.x** (the vector and nearest-join examples require it).


## 11. [OPTIONAL] ANSI mode and safe casts
Malformed numeric text can cause runtime errors when ANSI mode is enabled. `try_cast` produces NULL instead of failing; decide explicitly what to do with NULLs. This exercise is deliberately safe to run.


In [52]:
%livy.pyspark
dirty = spark.createDataFrame([('12',), ('unknown',), ('-666666666',), (None,)], ['raw'])
dirty.select('raw', F.expr('try_cast(raw AS INT)').alias('safe_int')).show()
print('ANSI:', spark.conf.get('spark.sql.ansi.enabled'))
# Experiment (expected failure with ANSI enabled): dirty.selectExpr('cast(raw AS INT)').show()


## 12. [OPTIONAL] JSON/VARIANT and semi-structured attributes
Yelp includes irregular attribute/hours objects. Construct miniature JSON strings, parse into a `VARIANT`, then extract safely. Requires Spark 4.x with VARIANT support.


In [54]:
%livy.pyspark
raw_business = spark.createDataFrame([
    ('b1', '{"RestaurantsPriceRange2":"2","WiFi":"free"}'),
    ('b2', '{"RestaurantsPriceRange2":"None","WiFi":null}'),
    ('b3', '{}')], ['business_id','attrs_json'])
variants = raw_business.withColumn('attrs', F.expr('parse_json(attrs_json)'))
variants.select('business_id',
    F.expr("try_variant_get(attrs, '$.RestaurantsPriceRange2', 'int')").alias('price')).show()


## 13. [OPTIONAL] Correct weighted averages after joins
If multiple reviews belong to each business, average the **reviews**, not the averages of individual businesses (unless weighting is intended). This is a common aggregation trap.


In [56]:
%livy.pyspark
ratings = spark.createDataFrame([('a',5),('a',5),('a',5),('b',1)], ['business_id','stars'])
by_b = ratings.groupBy('business_id').agg(F.avg('stars').alias('business_avg'))
print('Wrong if the goal is average over reviews:', by_b.agg(F.avg('business_avg')).first()[0])
print('Right:', ratings.agg(F.avg('stars')).first()[0])


## Example 13 — Aggregation Pitfall: Average of Averages

A common analytical mistake is calculating the **unweighted average of group averages** when the goal is to calculate the average across all original records.

This is particularly important when processing large datasets in multiple aggregation stages.

### 1. The Example

Consider four reviews belonging to two businesses:

| Business | Ratings | Number of reviews | Average |
|----------|---------|-------------------|---------|
| a | 5, 5, 5 | 3 | 5.0 |
| b | 1 | 1 | 1.0 |

Our code first calculates the average rating for each business:

```python
by_b = ratings.groupBy("business_id").agg(
    F.avg("stars").alias("business_avg")
)
```

Then it computes the average of these business averages:

```python
by_b.agg(F.avg("business_avg")).show()
```

Result: **3.0**

But calculating the average across all individual reviews:

```python
ratings.agg(F.avg("stars")).show()
```

produces **4.0**.

### 2. Why Are the Results Different?

**Average of business averages:**

`(5 + 1) / 2 = 3.0`

Each business contributes equally, regardless of its review count.

**Average over all reviews:**

`(5 + 5 + 5 + 1) / 4 = 4.0`

Each review contributes equally.

Neither result is inherently wrong: they represent different statistical quantities.

The mistake occurs when an unweighted average of group averages is used as a substitute for an average over the original records.

### 3. Correct Aggregation Using Weights

If we have already calculated business-level averages, we must retain the number of reviews in each group.

```python
by_b = ratings.groupBy("business_id").agg(
    F.count("*").alias("n"),
    F.avg("stars").alias("business_avg")
)

by_b.agg(
    (
        F.sum(F.col("business_avg") * F.col("n")) /
        F.sum("n")
    ).alias("overall_avg")
).show()
```

Result: **4.0**

The weighted average accounts for the number of records contributing to each intermediate average.

For a collection of groups, the overall average is:

`overall_avg = SUM(group_avg × group_count) / SUM(group_count)`

An even better approach is to retain partial sums and counts:

```python
by_b = ratings.groupBy("business_id").agg(
    F.sum("stars").alias("rating_sum"),
    F.count("stars").alias("rating_count")
)

by_b.agg(
    (
        F.sum("rating_sum") /
        F.sum("rating_count")
    ).alias("overall_avg")
).show()
```

Using `count("stars")` ensures that NULL ratings are excluded from the denominator, just as they are by `avg("stars")`.

### 4. Connection to Distributed Aggregation

Recall the two-phase aggregation strategy used by Spark:

1. **Partial aggregation:** Compute intermediate sums and counts per partition.
2. **Shuffle:** Redistribute the partial aggregates.
3. **Final aggregation:** Combine the sums and counts.

Spark does not simply average the partial averages.

Doing so would produce incorrect results whenever different partitions contain different numbers of records.

This is one reason `AVG()` requires intermediate aggregate state containing both a sum and a count.


**Always identify the unit of observation before calculating an average:** Are we averaging reviews, businesses, users, or geographic areas?

### Takeaways

- The average of averages is generally not equal to the overall average.
- Group averages must be weighted by the number of contributing observations.
- Partial sums and counts can be safely combined across partitions.
- `AVG()` excludes NULL values, so counts must refer to non-null observations.
- Always distinguish between equally weighting groups and equally weighting individual records.
- Correct aggregation semantics matter as much as distributed execution performance.

### Discussion Questions

1. Under what conditions does the average of group averages equal the overall average?
2. Why does Spark retain both sums and counts when computing distributed averages?
3. What happens if the business with only one review receives the same weight as a business with 10,000 reviews?
4. Why might averaging Yelp's `business.stars` field produce a different result from averaging individual review ratings?
5. Can an incorrectly weighted calculation still produce plausible-looking results?


## 14. [OPTIONAL] Geospatial top-k neighbors (toy points)
Compute spherical distance with built-in expressions—no Python UDF. The project's spatial query involves much more data, so a full cross join becomes expensive. The small example shows the logic of excluding self and applying top-k per source.


In [59]:
%livy.pyspark
places = spark.createDataFrame([
 ('a', 37.9755, 23.7348), ('b', 37.9838, 23.7275),
 ('c', 37.9715, 23.7257), ('d', 37.9900, 23.7500)], ['id','lat','lon'])
a, b = places.alias('a'), places.alias('b')
lat1, lat2 = F.radians(F.col('a.lat')), F.radians(F.col('b.lat'))
dlat = lat2-lat1; dlon=F.radians(F.col('b.lon')-F.col('a.lon'))
h = F.pow(F.sin(dlat/2),2)+F.cos(lat1)*F.cos(lat2)*F.pow(F.sin(dlon/2),2)
d = 2*F.lit(6371.0)*F.asin(F.sqrt(h))
pairs = a.crossJoin(b).where(F.col('a.id') != F.col('b.id')).select(
    F.col('a.id').alias('source'), F.col('b.id').alias('neighbor'), d.alias('km'))
w2=Window.partitionBy('source').orderBy('km','neighbor')
pairs.withColumn('rank',F.row_number().over(w2)).where('rank <= 2').orderBy('source','rank').show()


## Example 14 — Geospatial Top-k Nearest Neighbors

Many analytical applications need to identify geographically nearby objects.

Examples include:

- Finding nearby restaurants, shops or facilities.
- Measuring accessibility to public services.
- Comparing the characteristics of neighboring businesses.
- Analyzing spatial relationships between observations.

This example finds the **two nearest neighbors of each location** using Spark DataFrames, the Haversine distance formula and window functions.

### 1. Generating Candidate Pairs

```python
a, b = places.alias("a"), places.alias("b")

pairs = a.crossJoin(b).where(
    F.col("a.id") != F.col("b.id")
)
```

A **cross join**, or Cartesian product, combines every record from the first dataset with every record from the second.

For 4 locations:

- Total pairs: `4 × 4 = 16`
- Self-pairs: `4`
- Valid candidate pairs: `16 - 4 = 12`

The self-pair filter excludes comparisons such as `(a, a)`.

**Important:** The pairs are directed: `(a, b)` and `(b, a)` are distinct records, even though their geographic distance is identical.

### 2. Haversine Distance

The Haversine formula estimates the shortest distance along the surface of a sphere between two geographic coordinates.

It uses:

- Latitude and longitude in **radians**.
- Differences in latitude and longitude.
- Trigonometric functions.
- The Earth's approximate mean radius of **6,371 km**.

Our implementation uses built-in Spark functions:

```python
lat1 = F.radians(F.col("a.lat"))
lat2 = F.radians(F.col("b.lat"))

dlat = lat2 - lat1
dlon = F.radians(F.col("b.lon") - F.col("a.lon"))

h = (
    F.pow(F.sin(dlat / 2), 2)
    + F.cos(lat1) * F.cos(lat2)
    * F.pow(F.sin(dlon / 2), 2)
)

d = 2 * F.lit(6371.0) * F.asin(F.sqrt(h))
```

This computes **great-circle distance**, not actual driving or walking distance.

![Great-circle distance](figures/distance.png)

Using native Spark SQL functions is generally preferable to implementing this calculation as a Python UDF, because Spark can evaluate the expressions within its optimized execution engine.

### 3. Ranking Neighbors

After calculating the distance for each candidate pair, define a window:

```python
w2 = Window.partitionBy("source") \
           .orderBy("km", "neighbor")
```

This performs:

1. Logical grouping by source location.
2. Ascending ordering by geographic distance.
3. Deterministic tie-breaking by neighbor ID.

Assign ranks and retain the two closest locations:

```python
pairs.withColumn(
    "rank", F.row_number().over(w2)
).where(
    F.col("rank") <= 2
)
```

Example output:

| Source | Neighbor | Distance (km) | Rank |
|--------|----------|---------------|------|
| a | c | 0.913 | 1 |
| a | b | 1.123 | 2 |
| b | a | 1.123 | 1 |
| b | c | 1.377 | 2 |
| c | a | 0.913 | 1 |
| c | b | 1.377 | 2 |
| d | b | 2.089 | 1 |
| d | a | 2.091 | 2 |

Notice that nearest-neighbor relationships are not necessarily symmetric.

For example, `a` is among the two nearest neighbors of `d`, but `d` is not among the two nearest neighbors of `a`.

### 4. Why Is the Cartesian Join Expensive?

For N locations, excluding self-pairs gives:

`N × (N - 1)` candidate pairs.

| Locations | Candidate pairs |
|-----------|-----------------|
| 4 | 12 |
| 1,000 | 999,000 |
| 10,000 | 99,990,000 |
| 50,000 | 2,499,950,000 |

The number of candidate comparisons grows quadratically:

`O(N²)`

This remains true even if we only need the 2, 5 or 10 nearest neighbors of each location.

Furthermore, ranking candidates within each source group requires distributed grouping and sorting.

The naive implementation can therefore become extremely expensive for large datasets.

### 5. Inspecting the Physical Plan

Construct the final query as a DataFrame:

```python
nearest = (
    pairs.withColumn(
        "rank", F.row_number().over(w2)
    )
    .where(F.col("rank") <= 2)
)

nearest.explain(mode="formatted")
```

Look for:

- `CartesianProduct` or another applicable Cartesian join strategy.
- `Project` for the distance calculation.
- `Exchange` for redistributing data by source.
- `Sort` for ordering neighbors by distance.
- `Window` for computing row numbers.
- `Filter` for retaining the top two neighbors.

The exact join strategy depends on input size estimates, hints and Spark configuration.

**Question:** Why can selecting only two nearest neighbors still require evaluating many candidate pairs?

Because the top-k filter is applied after candidate generation and distance computation in this straightforward implementation.

### 6. Improving Scalability

For larger datasets, possible optimization techniques include:

**Spatial filtering**

Exclude candidates that are clearly too far away before computing expensive Haversine distances.

**Spatial partitioning**

Organize locations into geographic regions so that nearby points can be searched more efficiently.

**Spatial indexes**

Use data structures that help identify promising nearby candidates without examining every point.

**Specialized nearest-neighbor operators**

Modern query engines can provide dedicated operators for nearest-neighbor search.

These approaches can significantly reduce the number of candidate comparisons, although their effectiveness and correctness depend on the implementation.

### 7. Connection to the Semester Assignment

Query 4 requires identifying the **five nearest restaurants** to each open restaurant in Pennsylvania.

The general procedure is:

1. Filter the dataset to eligible restaurants.
2. Generate candidate restaurant pairs.
3. Exclude comparisons of a restaurant with itself.
4. Calculate geographic distances using Haversine.
5. Rank neighboring restaurants independently for each source.
6. Retain the five closest neighbors.
7. Aggregate neighborhood statistics.

The project also introduces the specialized `NEAREST BY` functionality available in Spark 4.2.

Our Spark 3.4 implementation demonstrates the conventional relational approach, making it possible to compare the physical execution strategies later.

### 8. Experiment: Change k

Modify the final filter:

```python
nearest.where(F.col("rank") <= 1)
```

Then try selecting three neighbors.

**Question:** Does reducing k from 3 to 1 eliminate the cost of generating all candidate pairs in the naive implementation?

No. The candidate generation and distance calculation still occur before selecting the nearest results.

### Takeaways

- A self-cross-join generates all possible location pairs.
- Self-pairs must be excluded when searching for other neighbors.
- Haversine calculates great-circle distance from geographic coordinates.
- Window functions can rank candidate neighbors independently for each source.
- Deterministic tie-breaking makes top-k results reproducible.
- Nearest-neighbor relationships are not necessarily symmetric.
- Naive all-pairs distance computation has quadratic complexity.
- Physical-plan inspection helps identify Cartesian products, shuffles and sorting costs.
- Specialized spatial processing techniques can improve scalability.

### Discussion Questions

1. Why does a cross join of N locations produce N² pairs?
2. Why do we exclude self-pairs?
3. Why must latitude and longitude be converted to radians?
4. Why is Haversine distance different from driving distance?
5. Why might `(a, b)` appear in the top-k results while `(b, a)` does not?
6. Why is a Cartesian join potentially dangerous for large datasets?
7. How could spatial filtering reduce the number of candidate comparisons?
8. Why is using native Spark functions generally preferable to Python UDFs for distance calculations?


## 15. [OPTIONAL] Spark 4.2 vector cosine similarity and `NEAREST BY`
Embeddings must be typed as `ARRAY<FLOAT>` for Spark's vector functions. `EXACT NEAREST` currently computes exact rankings rather than a magic sublinear index lookup. A nearest join can return each row itself; request k+1 and exclude self when using a self-join.


In [62]:
%livy.pyspark
vectors = spark.sql('''SELECT * FROM VALUES
 ('pizza', array(1.0F, 0.0F, 0.0F)),
 ('pasta', array(0.9F, 0.1F, 0.0F)),
 ('coffee', array(0.0F, 1.0F, 0.0F)),
 ('tea', array(0.0F, 0.9F, 0.1F)) AS t(word, vec)''')
vectors.createOrReplaceTempView('toy_vectors')
spark.sql('''SELECT q.word AS query_word, t.word AS neighbor,
       vector_cosine_similarity(q.vec, t.vec) AS similarity
FROM (SELECT * FROM toy_vectors WHERE word = 'pizza') q
JOIN toy_vectors t EXACT NEAREST 3 BY SIMILARITY vector_cosine_similarity(q.vec, t.vec)''').show()


## 16. [OPTIONAL] Benchmark hygiene and student exercises
**Benchmark hygiene:** time actions that actually compute the requested output, repeat runs, document resources/configuration, and compare physical plans and Spark UI metrics. Repeating a measurement in a fresh Spark application limits application-level cache carryover, but does **not** necessarily remove OS page cache.

**Try independently:**
1. Find the top 2 salaries per division, not department.
2. Compare `left_anti` and `left` + `isNull` for missing dimension records.
3. Add a `year` filter to the Parquet example and identify `PartitionFilters` in `explain()`.
4. Increase synthetic partitions from 2 to 8 and compare tasks/stages (not just elapsed time).
5. Explain why a nested-loop spatial self-join scales quadratically.

**Project mapping:** ETL → section 8; Query 1 → sections 3/6/9; Query 2 → sections 4/5; Query 3 → sections 4/6; Query 4 → section 14; Query 5 → sections 8/11/12/13; Query 6 → section 15; benchmarking → sections 7/9/10/16.
